# 인수 분해 형태와 스킵-트라이그램 버그
## 섹션: 인수 분해의 한계와 버그

> **대상**: 트랜스포머 기초를 막 배운 초보자  
> **핵심 질문**: 어텐션 헤드가 두 패턴을 동시에 학습할 때, 왜 의도치 않은 패턴도 함께 생기는가?

이 노트북은 Anthropic의 **"A Mathematical Framework for Transformer Circuits"** 논문에서  
발견한 흥미로운 **구조적 버그**를 재현합니다.  
모델이 왜 "틀린" 예측을 하는지, 수식과 숫자로 직접 추적합니다.

---
**📌 메타 정보**
- Tutorial ID: `adv-7-1` / Tutorial: 인수 분해 형태와 스킵-트라이그램 버그
- Section ID: `adv-7-1-1` / Section: 인수 분해의 한계와 버그

## 📚 학습 목표 & 사전 지식

### 이 노트북을 마치면 다음을 이해할 수 있습니다

| # | 개념 | 핵심 질문 |
|---|------|-----------|
| 1 | **인수 분해 형태** (Factored Form) | 어텐션이 왜 QK × OV 두 행렬의 곱으로 나뉘는가? |
| 2 | **스킵-트라이그램** (Skip-Trigram) | `"A ... B → C"` 패턴이 어떻게 동작하는가? |
| 3 | **교차 오염 버그** | 같은 source를 공유하면 왜 엉뚱한 예측도 생기는가? |
| 4 | **해석성** (Interpretability) | 이 분석이 AI 이해에 왜 중요한가? |

### 전제 지식 (없어도 괜찮습니다!)

- **Softmax**: 점수 벡터를 확률 분포로 만드는 함수 → Cell 4에서 직접 구현하며 설명
- **행렬 인덱싱**: `A[i, j]`처럼 2D 배열에서 원소를 선택하는 방법
- **어텐션 개념**: "토큰이 다른 토큰을 주목한다"는 아이디어를 들어본 적 있으면 충분

> 💡 **읽는 팁**: 각 셀의 마크다운 설명을 먼저 읽고, 코드를 실행한 뒤 출력 숫자를 확인하세요.  
> 처음엔 모든 걸 이해하려 하기보다 "어디서 숫자가 어떻게 바뀌는지" 흐름을 따라가세요.

In [ ]:
# =====================================================================
# 코드 읽는 법 (이 셀은 실행해도 아무것도 안 합니다 — 안내만)
# =====================================================================
#
# 이 노트북의 구성:
#   Part 1 │ 어휘·행렬 초기화      — 실험 무대 세팅
#   Part 2 │ 패턴 주입             — 모델이 학습한 상태 재현
#   Part 3 │ 스킵-트라이그램 점수  — 올바른 패턴 확인
#   Part 4 │ 버그 발견             — 교차 오염 4가지 조합
#   Part 5 │ 전체 예측 파이프라인  — 실제 확률 계산
#   Part 6 │ 요약 & 연습 문제
#
# 핵심 변수 미리 보기:
#   V         어휘 크기 (= 7)
#   vocab     단어 목록 ['keep','in','at','mind','bay','on','hand']
#   C_QK      Query-Key  행렬 (V×V)  — "어디를 볼까?"
#   C_OV      Output-Val 행렬 (V×V)  — "무엇을 예측할까?"
#
# 추천 실행 방법:
#   Shift+Enter 로 한 셀씩 실행하면서 출력을 확인하세요.
# =====================================================================
print('안내 셀 — 다음 셀부터 실행하세요 (Shift+Enter)')

In [ ]:
# =====================================================================
# [셀 4] 라이브러리 임포트 & 핵심 헬퍼 함수
# =====================================================================
# numpy   : 행렬·벡터 계산 표준 라이브러리
# itertools: 조합(combination) 생성
#
# PyTorch / TensorFlow 없이 numpy 만으로 어텐션 수식을 구현합니다.
# =====================================================================

import numpy as np
import itertools

# ─── 출력 보조 ────────────────────────────────────────────────────
def divider(title='', char='=', width=62):
    # 섹션 구분선 출력 헬퍼
    if title:
        pad = max(0, width - len(title) - 2)
        l = pad // 2
        r = pad - l
        print(char * l + ' ' + title + ' ' + char * r)
    else:
        print(char * width)

# ─── Softmax 함수 ─────────────────────────────────────────────────
def softmax(x):
    # Softmax: 점수(숫자 벡터) → 확률 분포 (모두 양수, 합=1)
    #
    # 수식: softmax(x_i) = exp(x_i) / Σ exp(x_j)
    #
    # [수치 안정화] 왜 max(x)를 먼저 빼는가?
    #   x 값이 크면 exp(x)가 매우 커져 overflow 발생 가능
    #   → 최댓값을 빼도 분자·분모가 같은 값으로 나뉘므로 결과 동일
    #   → 단지 계산 안전성을 위한 트릭
    #
    # 예시:
    #   입력: [1.0, 2.0, 3.0]
    #   출력: [0.09, 0.24, 0.67]  (합 = 1.0)
    x = np.asarray(x, dtype=float)
    e = np.exp(x - np.max(x))
    return e / e.sum()

# ─── 동작 확인 ────────────────────────────────────────────────────
print('✅ 라이브러리 로드 완료!')
print(f'   numpy 버전: {np.__version__}')
print()

print('[softmax 함수 동작 예시]')
ex_scores = np.array([1.0, 2.0, 3.0])
ex_probs  = softmax(ex_scores)
print(f'   입력 점수 : {ex_scores}')
print(f'   출력 확률 : {np.round(ex_probs, 3)}')
print(f'   확률 합계 : {ex_probs.sum():.1f}  ← 항상 1.0 이어야 함')
print()
print('숫자가 클수록 확률이 기하급수적으로 높아집니다.')
print('예: 점수 3이 점수 1보다 확률이 약 7.4배 높음')

## Part 1: 어텐션 메커니즘과 인수 분해 형태

### 어텐션 헤드가 하는 두 가지 일

트랜스포머의 단일 어텐션 헤드는 두 단계로 동작합니다:

```
[단계 1] 어디를 볼까?   →  W_QK 행렬
[단계 2] 무엇을 가져올까? → W_OV 행렬
```

| 행렬 | 이름 | 역할 | 비유 |
|------|------|------|------|
| **W_QK** | Query-Key | "어떤 과거 토큰에 집중할지" | 레이더로 신호 탐지 |
| **W_OV** | Output-Value | "그 토큰에서 무엇을 복사할지" | 선택된 파일에서 정보 추출 |

### 왜 "인수 분해(Factored)"인가?

단일 어텐션 헤드의 효과는 두 행렬의 **곱**으로 분해됩니다:

```
실효 가중치 = W_OV  ×  W_QK
              ↑           ↑
         "무엇을?"    "어디서?"
```

이 **분리(factored) 구조** 때문에, 아래에서 볼 버그가 필연적으로 발생합니다.

---
> **다음 셀**: 실험용 어휘(7개 단어)와 행렬을 설정합니다.

In [ ]:
# =====================================================================
# [셀 6] Part 1 — 실험용 어휘(Vocabulary) 설정
# =====================================================================
# 실제 GPT-4 등은 약 10만 개의 토큰을 사용하지만,
# 이 실험에서는 7개 단어로 압축합니다.
#
# 학습 목표 패턴 (영어 관용구):
#   ① "keep [다른 단어들] in mind"  →  명심하다
#   ② "keep [다른 단어들] at bay"   →  막아두다
#
# 두 패턴 모두 "keep"이 앞에 있고, "in/at"이 나중에 등장합니다.
# 중간에 다른 단어가 낄 수 있으므로 "스킵(skip)" 트라이그램!
# =====================================================================

divider('Part 1 — 어휘 & 행렬 설정')

vocab = ['keep', 'in', 'at', 'mind', 'bay', 'on', 'hand']
V     = len(vocab)   # 어휘 크기 = 7

# 각 단어의 인덱스를 변수로 저장 (코드 가독성을 위해)
keep_idx = vocab.index('keep')   # 0
in_idx   = vocab.index('in')     # 1
at_idx   = vocab.index('at')     # 2
mind_idx = vocab.index('mind')   # 3
bay_idx  = vocab.index('bay')    # 4
on_idx   = vocab.index('on')     # 5
hand_idx = vocab.index('hand')   # 6

print('단어 → 인덱스 매핑:')
for i, w in enumerate(vocab):
    mark = '  ★ (핵심 단어)' if w in ('keep','in','at','mind','bay') else ''
    print(f'  [{i}] {w!r}{mark}')

print(f'\n어휘 크기 V = {V}')
print()
print('[학습 목표 패턴]')
print('  ① keep ... in → mind   ("keep in mind" 패턴)')
print('     해석: "in" 이 "keep" 을 발견하고, "mind" 를 예측')
print()
print('  ② keep ... at → bay    ("keep at bay" 패턴)')
print('     해석: "at" 이 "keep" 을 발견하고, "bay" 를 예측')
print()
print('⚠️  두 패턴이 공통 source "keep" 을 공유합니다 — 이게 핵심!')

## Part 1 계속: C_QK와 C_OV 행렬 구조

### C_QK[query, source] — "어디를 볼까?"

```
C_QK 행렬 (V × V)

         keep  in   at  mind  bay   on  hand
keep  [  ···   ···  ···  ···  ···   ···  ···  ]
in    [  ???   ···  ···  ···  ···   ···  ···  ]  ← 'in'이 'keep'을 얼마나 주목?
at    [  ???   ···  ···  ···  ···   ···  ···  ]  ← 'at'이 'keep'을 얼마나 주목?
mind  [  ···   ···  ···  ···  ···   ···  ···  ]
...
```

`C_QK[in, keep] = 3.0` 이라면 → `in`이라는 단어가 나타날 때, 앞에 있는 `keep`에 강하게 집중

### C_OV[source, next] — "무엇을 예측할까?"

```
C_OV 행렬 (V × V)

         keep  in   at  mind  bay   on  hand
keep  [  ···   ···  ···  ???  ???   ···  ···  ]  ← 'keep'이 'mind'/'bay' 예측에 기여?
in    [  ···   ···  ···  ···  ···   ···  ···  ]
...
```

`C_OV[keep, mind] = 3.0` 이라면 → `keep`을 주목했을 때, 다음 단어로 `mind`를 강하게 예측

### 두 행렬을 합친 스킵-트라이그램 점수

```
score("source ...query → next") = C_QK[query, source]  ×  C_OV[source, next]
                                      ↑                        ↑
                                  "주목 강도"               "예측 강도"
```

이 **곱(×)** 구조가 바로 오늘 버그의 핵심 원인입니다.

In [ ]:
# =====================================================================
# [셀 8] C_QK & C_OV 행렬 초기화
# =====================================================================
# 처음에는 작은 무작위 값(잡음)으로 채웁니다.
# 실제 신경망 학습 전 가중치 초기화와 비슷합니다.
#
# 값 범위: ±0.1 수준 (배경 잡음)
# 나중에 주입할 패턴 값: 3.0 (잡음의 약 30배 → 압도적 신호)
# =====================================================================

np.random.seed(42)   # 시드 고정: 실행할 때마다 같은 무작위 값

C_QK = np.random.randn(V, V) * 0.1
C_OV = np.random.randn(V, V) * 0.1

print('C_QK (Query-Key 행렬):')
print(f'  shape     : {C_QK.shape}  (행 = query 단어, 열 = source 단어)')
print(f'  값 범위   : [{C_QK.min():.3f}, {C_QK.max():.3f}]  ← 배경 잡음, 패턴 없음')
print()
print('C_OV (Output-Value 행렬):')
print(f'  shape     : {C_OV.shape}  (행 = source 단어, 열 = next 단어)')
print(f'  값 범위   : [{C_OV.min():.3f}, {C_OV.max():.3f}]  ← 배경 잡음, 패턴 없음')
print()

# 아직 아무것도 주입하지 않은 상태의 keep 관련 값 미리 보기
print('[패턴 주입 전] keep 관련 값 미리 보기:')
print(f'  C_QK[in_idx, keep_idx]   = {C_QK[in_idx, keep_idx]:.4f}  (← 잡음 수준)')
print(f'  C_OV[keep_idx, mind_idx] = {C_OV[keep_idx, mind_idx]:.4f}  (← 잡음 수준)')
print()
print('다음 셀에서 이 값들을 3.0으로 "주입"합니다.')

## Part 2: 원하는 패턴 주입

### 실제 훈련과 이 실험의 차이

실제 트랜스포머 훈련에서는 수백만 개의 문장을 보여주고,  
경사하강법(gradient descent)으로 행렬 값이 **자동**으로 학습됩니다.

이 실험에서는 "이미 학습이 끝난 상태"를 **직접 재현**합니다:  
원하는 셀에 값 `3.0`을 넣으면, 해당 패턴이 강하게 학습된 것처럼 동작합니다.

### 왜 3.0인가?

| 위치 | 값 | 의미 |
|------|-----|------|
| 배경 잡음 | ≈ ±0.1 | 아무것도 학습 안 된 기본 상태 |
| 패턴 신호 | 3.0 | 잡음보다 **30배** 강한 신호 |

Softmax를 통과하면 3.0짜리 신호가 0.1짜리 잡음을 압도해 거의 100% 확률이 됩니다.

---
> **다음 셀**: 두 관용구 패턴을 행렬에 직접 주입합니다.

In [ ]:
# =====================================================================
# [셀 10] Part 2 — 패턴 주입
# =====================================================================
# 각 패턴은 QK 한 곳 + OV 한 곳, 총 두 셀을 설정합니다.
#
# 패턴 A: "keep ... in → mind"
#   C_QK[in,   keep] = 3.0  → 'in'이 'keep'을 주목
#   C_OV[keep, mind] = 3.0  → 'keep'이 'mind'를 예측
#
# 패턴 B: "keep ... at → bay"
#   C_QK[at,   keep] = 3.0  → 'at'이 'keep'을 주목
#   C_OV[keep, bay]  = 3.0  → 'keep'이 'bay'를 예측
# =====================================================================

divider('Part 2 — 패턴 주입')

print()
print('══ 패턴 A: "keep ... in → mind" (명심하다) ══')
print()
print('[단계 A-1] "in"이 "keep"을 강하게 주목하도록 설정')
C_QK[in_idx, keep_idx] = 3.0
print(f'  C_QK[in_idx, keep_idx] = {C_QK[in_idx, keep_idx]:.1f}')
print(f'  해석: 문장에서 "in"을 만나면, 앞에 있는 "keep"에 집중합니다.')
print()
print('[단계 A-2] "keep"을 보면 다음 단어로 "mind"를 예측하도록 설정')
C_OV[keep_idx, mind_idx] = 3.0
print(f'  C_OV[keep_idx, mind_idx] = {C_OV[keep_idx, mind_idx]:.1f}')
print(f'  해석: "keep"에 집중하면, 다음 단어는 "mind"가 됩니다.')

print()
print('──────────────────────────────────────────────────')
print()
print('══ 패턴 B: "keep ... at → bay" (막아두다) ══')
print()
print('[단계 B-1] "at"이 "keep"을 강하게 주목하도록 설정')
C_QK[at_idx, keep_idx] = 3.0
print(f'  C_QK[at_idx, keep_idx] = {C_QK[at_idx, keep_idx]:.1f}')
print(f'  해석: 문장에서 "at"을 만나면, 앞에 있는 "keep"에 집중합니다.')
print()
print('[단계 B-2] "keep"을 보면 다음 단어로 "bay"를 예측하도록 설정')
C_OV[keep_idx, bay_idx] = 3.0
print(f'  C_OV[keep_idx, bay_idx] = {C_OV[keep_idx, bay_idx]:.1f}')
print(f'  해석: "keep"에 집중하면, 다음 단어는 "bay"가 됩니다.')

print()
print('=' * 50)
print()
print('패턴 주입 요약:')
print('  C_QK: in  → keep = 3.0   (in이  keep 주목)')
print('        at  → keep = 3.0   (at이  keep 주목)')
print('  C_OV: keep → mind = 3.0  (keep이 mind 예측)')
print('        keep → bay  = 3.0  (keep이 bay  예측)')
print()
print('⚠️  "in"과 "at"이 모두 같은 source "keep"을 향합니다.')
print('    "keep"은 "mind"와 "bay"를 모두 예측합니다.')
print('    이 두 사실의 조합이 버그를 만들어냅니다...')

## Part 3: 스킵-트라이그램(Skip-Trigram)이란?

### n-그램 개념 비교

| 용어 | 예시 | 설명 |
|------|------|------|
| **유니그램** | `"mind"` | 단어 1개 |
| **바이그램** | `"in mind"` | 연속 2개 |
| **트라이그램** | `"keep in mind"` | 연속 3개 |
| **스킵-트라이그램** | `"keep [???] in → mind"` | 중간에 다른 단어가 있어도 OK |

### 실제 문장 예시

```
문장:  "I need to  keep  this  in   ___"
                   ↑            ↑    ↑
                source        query  next(예측)
                (keep)        (in)   (mind)
```

```
문장:  "Try to   keep  the problem  at   ___"
                 ↑                  ↑    ↑
              source               query  next
              (keep)               (at)  (bay)
```

어텐션 헤드가 이 패턴을 포착하는 과정:
1. 현재 위치의 단어(`in`)가 **query**로서 이전 단어들을 스캔
2. 앞에서 `keep`을 발견 → `C_QK[in, keep] = 3.0` 이므로 강하게 집중
3. `keep`의 `C_OV` 값을 읽어 다음 단어 예측 → `C_OV[keep, mind] = 3.0`

### 스킵-트라이그램 점수 공식

```python
score(query, source, next) = C_QK[query, source] × C_OV[source, next]
```

이 **단순한 곱셈**이 핵심 버그의 원인입니다. 왜 그런지 다음 셀에서 확인합니다.

In [ ]:
# =====================================================================
# [셀 12] Part 3 — 스킵-트라이그램 점수 함수 & 의도한 패턴 확인
# =====================================================================

def skip_trigram_score(query_idx, source_idx, next_idx, verbose=False):
    # 스킵-트라이그램 점수 계산
    #
    # 공식: score = C_QK[query, source]  ×  C_OV[source, next]
    #
    # 두 값을 곱하는 이유:
    #   - QK: "query가 source를 얼마나 주목하는가?" (주목 강도)
    #   - OV: "source가 next를 얼마나 예측하는가?" (예측 강도)
    #   - 둘 다 높아야 최종 점수가 높음  →  AND 조건처럼 동작
    #
    # 예) score(in, keep, mind) = QK[in,keep] × OV[keep,mind]
    #                           = 3.0 × 3.0 = 9.0
    
    qk_val = C_QK[query_idx, source_idx]
    ov_val = C_OV[source_idx, next_idx]
    total  = qk_val * ov_val
    
    if verbose:
        q_w = vocab[query_idx]
        s_w = vocab[source_idx]
        n_w = vocab[next_idx]
        print(f'  패턴: "{s_w} ... {q_w} → {n_w}"')
        print(f'    C_QK[{q_w!r:5}, {s_w!r:6}] = {qk_val:6.2f}  ← 주목 강도')
        print(f'    C_OV[{s_w!r:5}, {n_w!r:6}] = {ov_val:6.2f}  ← 예측 강도')
        print(f'    점수 = {qk_val:.2f} × {ov_val:.2f} = {total:.2f}')
    
    return total

divider('Part 3 — 스킵-트라이그램 점수')

print()
print('[✓ 의도한 패턴 A] "keep ... in → mind"')
s_A = skip_trigram_score(in_idx, keep_idx, mind_idx, verbose=True)
print()

print('[✓ 의도한 패턴 B] "keep ... at → bay"')
s_B = skip_trigram_score(at_idx, keep_idx, bay_idx, verbose=True)

print()
print('─' * 50)
print(f'패턴 A 점수: {s_A:.1f}  ✓  (QK=3.0 × OV=3.0 = 9.0)')
print(f'패턴 B 점수: {s_B:.1f}  ✓  (QK=3.0 × OV=3.0 = 9.0)')
print()
print('두 패턴 모두 9.0으로 강하게 활성화됩니다.')
print()
print('그렇다면 이 공식이 만드는 다른 조합들은 어떨까요?')
print('→ 다음 셀에서 확인해봅니다...')

## Part 4: 버그 발견! — 교차 오염(Cross-Contamination)

### 문제의 핵심: 분리 가능성(Separability)

스킵-트라이그램 점수를 다시 봅시다:

```
score(query, source, next) = C_QK[query, source]  ×  C_OV[source, next]
                                   ↑                        ↑
                              query만 관련             next만 관련
```

`query`와 `next`는 **직접 연결되지 않고** `source`를 거쳐서만 간접 연결됩니다.

### 왜 이게 문제인가?

현재 상황:
- `C_QK[in,  keep] = 3.0`  → `in`이 `keep`을 주목
- `C_QK[at,  keep] = 3.0`  → `at`이 `keep`을 주목
- `C_OV[keep, mind] = 3.0` → `keep`이 `mind`를 예측
- `C_OV[keep, bay]  = 3.0` → `keep`이 `bay`를 예측

`source = 'keep'`을 중심으로 **4가지** 조합이 모두 가능합니다:

| query | source | next | 원하는가? |
|-------|--------|------|-----------|
| `in`  | `keep` | `mind` | ✓ 원함 |
| `in`  | `keep` | `bay`  | ✗ **버그!** |
| `at`  | `keep` | `mind` | ✗ **버그!** |
| `at`  | `keep` | `bay`  | ✓ 원함 |

이 4가지 점수가 **모두 동일**합니다. 모델은 구별하지 못합니다!

### 비유: 두 명의 비서

> **비서 A** (W_QK): "어떤 파일을 가져올지" 결정  
> **비서 B** (W_OV): "그 파일에서 무엇을 읽을지" 결정  
>
> 비서 A가 "keep 파일"을 선택하면,  
> 비서 B는 그 파일에서 "mind"와 "bay" **둘 다** 읽어버립니다.  
> 비서 A가 **왜** 그 파일을 선택했는지(`in` 때문인지 `at` 때문인지)를  
> 비서 B는 전혀 알지 못합니다!

In [ ]:
# =====================================================================
# [셀 14] Part 4 — 버그 발견: 모든 4가지 조합 점수 계산
# =====================================================================
# 두 패턴이 같은 source('keep')을 공유하기 때문에
# 의도하지 않은 교차 조합도 같은 점수를 받습니다.
# =====================================================================

divider('Part 4 — 버그 발견!')

print()
print('현재 행렬에 저장된 강한 신호 (3.0):')
print()
print('  C_QK:  in → keep  (점수 3.0)   C_OV:  keep → mind (점수 3.0)')
print('         at → keep  (점수 3.0)          keep → bay  (점수 3.0)')
print()
print('스킵-트라이그램 점수 = QK × OV 이므로,')
print('source="keep" 를 거치는 모든 (query, next) 조합:')
print()

queries = [(in_idx, 'in'), (at_idx, 'at')]
nexts   = [(mind_idx, 'mind'), (bay_idx, 'bay')]

hdr = f"  {'query':6s}  {'source':6s}  {'next':6s}  {'QK 점수':>8s}  {'OV 점수':>8s}  {'총 점수':>8s}  상태"
print(hdr)
print('  ' + '─' * 64)

for (q_idx, q_w), (n_idx, n_w) in itertools.product(queries, nexts):
    qk    = C_QK[q_idx, keep_idx]
    ov    = C_OV[keep_idx, n_idx]
    score = qk * ov
    
    intended = (q_w == 'in' and n_w == 'mind') or \
               (q_w == 'at' and n_w == 'bay')
    status   = '✓ 원함  ' if intended else '✗ 버그!'
    
    print(f'  {q_w:6s}  {"keep":6s}  {n_w:6s}  {qk:8.2f}  {ov:8.2f}  {score:8.2f}  {status}')

print()
print('=' * 66)
print()
print('⚠️  핵심 발견:')
print('   4가지 조합의 점수가 모두 9.00으로 완전히 동일합니다!')
print()
print('   모델 입장에서:')
print('   "keep...in→mind" (맞음)과 "keep...in→bay" (틀림)을')
print('   똑같이 좋은 예측으로 간주합니다.')
print()
print('이것이 1-레이어 인수 분해 구조의 근본적 한계입니다.')

In [ ]:
# =====================================================================
# [셀 15] 행렬 히트맵 — 버그의 원인을 눈으로 확인
# =====================================================================
# 행렬 전체를 출력해서 강한 신호가 어디에 심어졌는지 봅니다.
# 잡음 수준(|값| < 0.3)은 '···'으로 생략합니다.
# =====================================================================

def print_heatmap(mat, row_labels, col_labels, title, highlights=None):
    # 행렬을 텍스트 히트맵으로 시각화
    # highlights: (row, col) 튜플 리스트 — 강조 표시할 셀 (★)
    COL_W = 8
    print()
    print(title)
    
    # 헤더 행
    hdr = ' ' * 7 + ''.join(f'{lbl:>{COL_W}s}' for lbl in col_labels)
    print(hdr)
    print(' ' * 7 + '─' * (COL_W * len(col_labels)))
    
    for i, r_lbl in enumerate(row_labels):
        row_str = f'{r_lbl:>6s} │'
        for j in range(len(col_labels)):
            val = mat[i, j]
            if highlights and (i, j) in highlights:
                cell = f' ★{val:4.1f}'   # 별표: 강조
            elif abs(val) < 0.3:
                cell = '   ···'           # 잡음: 생략
            else:
                cell = f' {val:6.2f}'
            row_str += f'{cell:>{COL_W}s}'
        print(row_str)

highlights_qk = [(in_idx, keep_idx), (at_idx, keep_idx)]
highlights_ov = [(keep_idx, mind_idx), (keep_idx, bay_idx)]

print_heatmap(C_QK, vocab, vocab,
              'C_QK [행=query, 열=source] — "어디를 볼까?"',
              highlights=highlights_qk)
print('  ★ 표시 = 강한 신호(3.0)가 심어진 셀')
print('  → "keep" 열에 in행, at행 두 군데에 ★')

print_heatmap(C_OV, vocab, vocab,
              'C_OV [행=source, 열=next] — "무엇을 예측할까?"',
              highlights=highlights_ov)
print('  ★ 표시 = 강한 신호(3.0)가 심어진 셀')
print('  → "keep" 행에 mind열, bay열 두 군데에 ★')

print()
print('──────────────────────────────────────────────────────────')
print('  C_QK "keep" 열  : ★ 2개 신호  (in, at  → keep)')
print('  C_OV "keep" 행  : ★ 2개 신호  (keep → mind, bay)')
print()
print('  점수 = QK × OV = "2개 신호" × "2개 신호" = 4가지 조합')
print('  → 원하는 2개 + 원하지 않는 2개 = 모두 점수 9.0')
print('  → 이것이 버그의 기하학적 원인!')

## Part 4 계속: 수학적으로 왜 불가피한가?

### 공식 분석

```
score(query, source='keep', next)
    = C_QK[query, 'keep']  ×  C_OV['keep', next]
```

두 인자를 분리해서 봅니다:

| 인자 | 가능한 값 |
|------|-----------|
| `C_QK[query, 'keep']` | `in→3.0`, `at→3.0`, 기타≈0 |
| `C_OV['keep', next]`  | `mind→3.0`, `bay→3.0`, 기타≈0 |

`query`와 `next`가 **서로 독립적으로** `source`에 연결되어 있습니다.

따라서 `query`가 무엇이든 (`in` 또는 `at`), `C_OV['keep', ·]`가 반환하는 값은 **항상 동일**합니다.  
즉, `query`의 정보가 `next` 예측에 영향을 줄 **경로가 없습니다**.

### 이 버그를 고칠 수 없는 이유

```
단일 어텐션 레이어에서:
  score = QK[query, source]  ×  OV[source, next]
  
이 구조에서 query → next 의 직접 경로는 없음
→ source가 같으면 query와 무관하게 next 예측이 동일
→ 학습 데이터를 늘려도, 더 오래 훈련해도 해결 불가
→ 아키텍처 문제 (데이터·하이퍼파라미터 문제 아님)
```

### 해결책

| 방법 | 원리 |
|------|------|
| **2-레이어 이상** | 레이어 간 구성(Composition)으로 더 세밀한 패턴 학습 가능 |
| **여러 어텐션 헤드** | 헤드마다 다른 source를 사용하도록 분담 |
| **실제 GPT** | 수십 레이어 + 수십 헤드로 이 한계를 사실상 극복 |

> 이 버그가 **사소해 보여도** 중요한 이유: 모델이 틀리는 이유를 수학적으로 추적하는  
> 해석성(Interpretability) 연구의 초기 성과 중 하나입니다.

In [ ]:
# =====================================================================
# [셀 17] Part 5 — 전체 어텐션 예측 파이프라인
# =====================================================================
# 스킵-트라이그램 점수를 넘어, 실제 어텐션처럼
# 1) QK 점수 계산 → 2) Softmax → 3) 가중합 → 4) Softmax → 확률
# 4단계를 거치는 완전한 예측을 구현합니다.
# =====================================================================

def simulate_attention(query_word, context_words, verbose=True):
    # 단순화된 단일 어텐션 헤드 시뮬레이션
    #
    # 입력:
    #   query_word   : 현재 예측 위치의 단어 (예: 'in')
    #   context_words: 이전 등장 단어들    (예: ['keep'])
    #
    # 반환:
    #   probs (np.array, shape=[V]): 각 어휘 단어의 예측 확률
    
    q_idx  = vocab.index(query_word)
    c_idxs = [vocab.index(w) for w in context_words]
    
    if verbose:
        divider(f'쿼리: "{query_word}" | 이전 단어: {context_words}', '-', 56)
    
    # ── STEP 1: 어텐션 점수 계산 ────────────────────────────────
    # query가 각 context 단어에 대해 C_QK 값을 읽음
    # → "어느 단어에 집중할지" 후보 점수
    attn_scores = np.array([C_QK[q_idx, c] for c in c_idxs])
    
    if verbose:
        print()
        print('  [STEP 1] QK 어텐션 점수 (Softmax 이전 원점수):')
        for w, sc in zip(context_words, attn_scores):
            print(f'    C_QK[{query_word!r:5}, {w!r:6}] = {sc:7.3f}')
    
    # ── STEP 2: Softmax → 어텐션 가중치 ────────────────────────
    # 점수를 확률로 변환 → 가중치 합 = 1.0
    # (여러 context가 있으면 모두를 어느 정도 고려)
    attn_weights = softmax(attn_scores)
    
    if verbose:
        print()
        print('  [STEP 2] 어텐션 가중치 (Softmax 후, 합=1.00):')
        for w, wt in zip(context_words, attn_weights):
            bar = '█' * max(0, int(wt * 24))
            print(f'    {w!r:8s}: {wt:.4f}  {bar}')
    
    # ── STEP 3: 가중합(Weighted Sum) → 출력 로짓 ───────────────
    # 각 context의 C_OV 행(row)을 어텐션 가중치로 더함
    # → "어떤 다음 단어를 예측해야 하는가"의 신호 벡터
    #
    # 수식: output = Σ_i (attn_weight[i] × C_OV[context_i, :])
    logits = np.zeros(V)
    for i, c_idx in enumerate(c_idxs):
        logits += attn_weights[i] * C_OV[c_idx]
    
    if verbose:
        top3 = np.argsort(logits)[::-1][:3]
        print()
        print('  [STEP 3] 출력 로짓 상위 3개 (Softmax 이전):')
        for idx in top3:
            print(f'    {vocab[idx]!r:8s}: {logits[idx]:.4f}')
    
    # ── STEP 4: Softmax → 최종 예측 확률 ───────────────────────
    # 로짓을 확률 분포로 변환
    probs = softmax(logits)
    
    if verbose:
        print()
        print('  [STEP 4] 최종 예측 확률 (확률 2% 이상만 표시):')
        sorted_ids = np.argsort(probs)[::-1]
        for idx in sorted_ids:
            if probs[idx] < 0.02:
                continue
            w   = vocab[idx]
            bar = '█' * max(1, int(probs[idx] * 30))
            
            if (query_word == 'in'  and w == 'mind') or \
               (query_word == 'at'  and w == 'bay'):
                tag = '  ← ✓ 올바른 예측'
            elif (query_word == 'in'  and w == 'bay') or \
                 (query_word == 'at'  and w == 'mind'):
                tag = '  ← ✗ 버그 예측 (원하지 않음)'
            else:
                tag = ''
            
            print(f'    {w!r:8s}: {probs[idx]:.4f}  {bar}{tag}')
    
    return probs

divider('Part 5 — 전체 예측 파이프라인')
print()
probs_in = simulate_attention('in', ['keep'])
print()
probs_at = simulate_attention('at', ['keep'])

In [ ]:
# =====================================================================
# [셀 18] 예측 결과 최종 비교
# =====================================================================

divider('결과 비교 요약')

print()
print('이상적인 결과 (버그가 없다면):')
print(f'  "keep ... in" → mind 확률 ≈ 1.0,  bay 확률 ≈ 0.0')
print(f'  "keep ... at" → bay  확률 ≈ 1.0,  mind 확률 ≈ 0.0')

print()
print('실제 결과:')
print(f'  "keep ... in" → mind = {probs_in[mind_idx]:.4f},  bay = {probs_in[bay_idx]:.4f}')
print(f'  "keep ... at" → bay  = {probs_at[bay_idx]:.4f},  mind = {probs_at[mind_idx]:.4f}')

print()
diff_in = abs(probs_in[mind_idx] - probs_in[bay_idx])
diff_at = abs(probs_at[bay_idx] - probs_at[mind_idx])
print(f'  mind/bay 확률 차이 ("in" 쿼리): {diff_in:.4f}')
print(f'  bay/mind 확률 차이 ("at" 쿼리): {diff_at:.4f}')
print()
print('차이가 거의 0 — 모델이 올바른 예측과 버그 예측을 구별 못함!]')

print()
print('─' * 62)
print()
print('이 버그의 특성:')
print('  ✗ 학습 데이터를 늘려도 해결 안 됨')
print('  ✗ 더 오래 훈련해도 해결 안 됨')
print('  ✗ 학습률 조정으로도 해결 안 됨')
print('  ✓ 레이어를 늘리면(2-레이어) 해결 가능')
print('  ✓ 서로 다른 source를 사용하는 패턴으로 분리하면 해결 가능')
print()
print('원인: 행렬 구조 (아키텍처) 자체의 수학적 제약')

## Part 6: 요약 — 이 실험에서 배운 것

### 핵심 3가지

**1. 인수 분해 형태란?**
```
단일 어텐션 헤드의 효과 = W_OV  ×  W_QK
                         ↑          ↑
                    "무엇을?"    "어디서?"
```
두 행렬이 분리(factored)되어 있어, 각자 독립적으로 동작합니다.

**2. 스킵-트라이그램이란?**
```
"source ... query → next" 형태의 3-토큰 패턴

점수 = C_QK[query, source] × C_OV[source, next]
```
어텐션이 떨어진 위치의 토큰 관계를 포착하는 방식.

**3. 교차 오염 버그란?**
```
두 패턴이 같은 source를 공유하면:
  QK: {query1, query2} → source
  OV: source → {next1, next2}

모든 (query, next) 조합이 동일한 점수 받음
= 원하는 패턴 + 원하지 않는 패턴이 구별 불가
```

### 이 연구가 중요한 이유

| 관점 | 의미 |
|------|------|
| **해석성** | 모델이 왜 틀리는지 수학적으로 추적 가능 |
| **예측** | 어떤 상황에서 모델이 실수할지 미리 알 수 있음 |
| **개선** | 구조적 한계를 알면 더 나은 아키텍처를 설계할 수 있음 |

> 논문 원문: *"이러한 버그는 사소해 보이지만,  
> 해석성 연구로 모델의 실패를 이해하는 초기 사례다."*  
> — Elhage et al., "A Mathematical Framework for Transformer Circuits", 2021

In [ ]:
# =====================================================================
# [셀 20] Part 6 — 연습 문제: 직접 실험해보세요!
# =====================================================================

divider('연습 문제')

print('''
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
연습 1: OV 값을 0으로 만들면 버그가 사라질까?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

C_OV[keep_idx, bay_idx] = 0.0 으로 설정하면?

  score('in', 'keep', 'bay') = QK × OV = 3.0 × 0.0 = 0.0  → 버그 사라짐?
  score('at', 'keep', 'bay') = 3.0 × 0.0 = 0.0             → at→bay 도 사라짐!

결과: 버그는 사라지지만 패턴 B("keep at bay")도 함께 사라집니다.
      버그를 지우려면 정상 패턴도 희생해야 합니다.

  실험 코드:
    C_OV_test = C_OV.copy()
    C_OV_test[keep_idx, bay_idx] = 0.0
    p = simulate_attention_custom('in', ['keep'], C_QK, C_OV_test)
    print(f'mind: {p[mind_idx]:.4f}, bay: {p[bay_idx]:.4f}')

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
연습 2: 패턴이 3개로 늘면 버그도 제곱으로 늘어난다
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

세 번째 패턴 "keep ... on → hand"를 추가하면?
  C_QK: in, at, on  → keep  (3개)
  C_OV: keep →  mind, bay, hand  (3개)
  → 가능한 조합: 3 × 3 = 9가지
  → 의도한 패턴: 3개, 버그: 6개!

  실험 코드:
    C_QK_v2 = C_QK.copy()
    C_OV_v2 = C_OV.copy()
    C_QK_v2[on_idx,   keep_idx] = 3.0
    C_OV_v2[keep_idx, hand_idx] = 3.0

    queries_v2 = [(in_idx,'in'), (at_idx,'at'), (on_idx,'on')]
    nexts_v2   = [(mind_idx,'mind'), (bay_idx,'bay'), (hand_idx,'hand')]
    for (qi,qw),(ni,nw) in itertools.product(queries_v2, nexts_v2):
        sc = C_QK_v2[qi, keep_idx] * C_OV_v2[keep_idx, ni]
        ok = (qw=='in' and nw=='mind') or (qw=='at' and nw=='bay') or (qw=='on' and nw=='hand')
        print(f'{qw} → {nw}: {sc:.1f}  {"✓" if ok else "✗ 버그"}')

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
연습 3: source가 다르면 버그가 없다!
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

만약 두 패턴이 서로 다른 source를 쓴다면?
  패턴 A: "hold ... in → mind"  (source='hold')
  패턴 B: "keep ... at → bay"   (source='keep')

  "in" 이 "keep" 을 보는 점수: C_QK[in, keep] ≈ 0.0  (잡음)
  → score("keep...in→bay") = 0.0 × 3.0 = 0.0  ← 버그 없음!

왜? source가 다르면 교차 오염이 일어날 "공통 경유지"가 없기 때문입니다.

질문: 이 경우 C_QK에서 어떤 셀을 설정해야 하나요? 직접 그려보세요.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
''')

# ── 연습 1 힌트: simulate_attention을 커스텀 행렬로 실행하는 함수 ──
def simulate_attention_custom(query_word, context_words, QK, OV, verbose=False):
    # simulate_attention 과 동일하지만, QK와 OV를 인자로 받음
    # 연습 1처럼 행렬 값을 바꿔 실험할 때 사용
    q_idx  = vocab.index(query_word)
    c_idxs = [vocab.index(w) for w in context_words]
    attn_w = softmax([QK[q_idx, c] for c in c_idxs])
    logits = sum(attn_w[i] * OV[c_idxs[i]] for i in range(len(c_idxs)))
    probs  = softmax(logits)
    if verbose:
        for idx in np.argsort(probs)[::-1]:
            if probs[idx] > 0.03:
                print(f'  {vocab[idx]!r}: {probs[idx]:.4f}')
    return probs

print()
print('[simulate_attention_custom 함수 준비 완료]')
print('위 연습 코드를 새 셀에 붙여넣고 직접 실행해보세요!')